In [2]:
import pandas as pd
import requests
import json
import os
import time

print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)
print("Setup OK")

Pandas: 2.3.3
Requests: 2.32.5
Setup OK


In [3]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:1b"

response = requests.get("http://localhost:11434/api/tags")

print("Status:", response.status_code)

if response.status_code == 200:
    print("Ollama connection: OK")

    print("\nInstalled models:")
    for model in response.json().get("models", []):
        print("-", model["name"])
else:
    print("Ollama connection FAILED")

Status: 200
Ollama connection: OK

Installed models:
- gemma3:1b


In [4]:
folder = r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"

csv_files = [
    f for f in os.listdir(folder)
    if f.lower().endswith(".csv")
]

print("CSV files found:")

for i, file in enumerate(csv_files):
    print(i, "->", file)

CSV files found:
0 -> Hexagon-Dataset - Economy + Technology + Miscellaneous.csv


In [7]:
# Replace 0 with the number shown for the Economy file

file_name = csv_files[0]

file_path = os.path.join(folder, file_name)

df = pd.read_csv(file_path)

print("File:", file_name)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 2 rows:")
display(df.head(2))

File: Hexagon-Dataset - Economy + Technology + Miscellaneous.csv
Rows: 133
Columns: 15

Columns:
['Question ID', 'Domain', 'Subdomain', 'English Question', 'Nepali Question', 'Gold Answer (English)', 'Gold Answer (Nepali)', 'Explanation', 'Source', 'Source ID', 'Source Title', 'Source URL', 'Source Type', 'Difficulty', 'Answer Type']

First 2 rows:


,Question ID,Domain,Subdomain,English Question,Nepali Question,Gold Answer (English),Gold Answer (Nepali),Explanation,Source,Source ID,Source Title,Source URL,Source Type,Difficulty,Answer Type
0,ECO001,Economy,Currency & Banking,What is the official currency of Nepal?,नेपालको आधिकारिक मुद्रा के हो?,Nepalese Rupee,नेपाली रुपैयाँ,"Nepal's legal tender is the Nepalese Rupee, is...",Nepal Rastra Bank,SRC001,Nepal Rastra Bank - Official Website,https://www.nrb.org.np,Government,Easy,Other
1,ECO002,Economy,Currency & Banking,What is the ISO 4217 currency code for the Nep...,नेपाली रुपैयाँको ISO 4217 मुद्रा कोड के हो?,NPR,NPR,NPR is the internationally recognized three-le...,ISO 4217 Standard,SRC002,ISO 4217 Currency Codes - International Organi...,https://www.iso.org/iso-4217-currency-codes.html,International Organization,Medium,Other


In [8]:
required_columns = [
    "Question ID",
    "English Question",
    "Nepali Question",
    "Gold Answer (English)",
    "Gold Answer (Nepali)"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    print("❌ Missing columns:", missing)
else:
    print("✅ All required columns exist")

✅ All required columns exist


In [9]:
dataset_context = df.to_csv(index=False)

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("Dataset characters:", len(dataset_context))
print("Approximate tokens:", len(dataset_context) // 4)

Number of rows: 133
Number of columns: 15
Dataset characters: 63298
Approximate tokens: 15824


In [10]:
CHUNK_SIZE = 150

all_results = []

num_chunks = (len(df) + CHUNK_SIZE - 1) // CHUNK_SIZE

print("Total questions:", len(df))
print("Chunk size:", CHUNK_SIZE)
print("Number of chunks:", num_chunks)

for chunk_no, start in enumerate(
    range(0, len(df), CHUNK_SIZE),
    start=1
):

    chunk = df.iloc[start:start + CHUNK_SIZE].copy()
    chunk_context = chunk.to_csv(index=False)

    print("\n" + "=" * 70)
    print(f"CHUNK {chunk_no}/{num_chunks}")
    print(f"Rows {start + 1} to {start + len(chunk)}")
    print("Context characters:", len(chunk_context))
    print("Approximate tokens:", len(chunk_context) // 4)
    print("=" * 70)

    for _, row in chunk.iterrows():

        question_id = str(row["Question ID"])

        # =====================================================
        # ENGLISH
        # =====================================================

        prompt_en = f"""
You are answering a benchmark question.

Use ONLY the information contained in the DATASET below.

Rules:
- Answer the question directly.
- Do not invent information.
- Do not use outside information.
- Do not repeat the question.
- Give only the answer.
- Do not add explanations or summaries.
- If the answer cannot be determined from the dataset, respond exactly:
NOT FOUND

DATASET:
{chunk_context}

QUESTION:
{row["English Question"]}

ANSWER:
"""

        try:
            response_en = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt_en,
                    "stream": False,
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=600
            )

            if response_en.status_code == 200:
                answer_en = response_en.json().get(
                    "response", ""
                ).strip()
            else:
                answer_en = f"REQUEST ERROR {response_en.status_code}"

        except Exception as e:
            answer_en = f"REQUEST ERROR: {e}"

        all_results.append({
            "Question ID": question_id,
            "Language": "English",
            "Question": row["English Question"],
            "LLM Answer": answer_en,
            "Dataset Answer": row["Gold Answer (English)"],
            "Manual Verdict": "",
            "Error Type": ""
        })

        print(f"{question_id} | English | done")


        # =====================================================
        # NEPALI
        # =====================================================

        prompt_ne = f"""
तपाईं benchmark प्रश्नको उत्तर दिइरहनुभएको छ।

तलको DATASET मा भएको जानकारी मात्र प्रयोग गर्नुहोस्।

नियमहरू:
- प्रश्नको सिधा उत्तर दिनुहोस्।
- जानकारी नबनाउनुहोस्।
- बाहिरी जानकारी प्रयोग नगर्नुहोस्।
- प्रश्न दोहोर्याउनुहोस्।
- केवल उत्तर दिनुहोस्।
- explanation वा summary नदिनुहोस्।
- DATASET बाट उत्तर निर्धारण गर्न नसके:
NOT FOUND
भन्नुहोस्।

DATASET:
{chunk_context}

QUESTION:
{row["Nepali Question"]}

ANSWER:
"""

        try:
            response_ne = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt_ne,
                    "stream": False,
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=600
            )

            if response_ne.status_code == 200:
                answer_ne = response_ne.json().get(
                    "response", ""
                ).strip()
            else:
                answer_ne = f"REQUEST ERROR {response_ne.status_code}"

        except Exception as e:
            answer_ne = f"REQUEST ERROR: {e}"

        all_results.append({
            "Question ID": question_id,
            "Language": "Nepali",
            "Question": row["Nepali Question"],
            "LLM Answer": answer_ne,
            "Dataset Answer": row["Gold Answer (Nepali)"],
            "Manual Verdict": "",
            "Error Type": ""
        })

        print(f"{question_id} | Nepali | done")

    # Save progress
    temp_file = os.path.join(
        folder,
        "Economy_Technology_Miscellaneous_Gemma3_1B_TEMP.csv"
    )

    pd.DataFrame(all_results).to_csv(
        temp_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Temporary progress saved:",
        len(all_results),
        "responses"
    )


print("\n" + "=" * 70)
print("GEMMA RUN COMPLETE")
print("=" * 70)

print("Total responses:", len(all_results))
print("Expected responses:", len(df) * 2)

Total questions: 133
Chunk size: 150
Number of chunks: 1

CHUNK 1/1
Rows 1 to 133
Context characters: 63298
Approximate tokens: 15824
ECO001 | English | done
ECO001 | Nepali | done
ECO002 | English | done
ECO002 | Nepali | done
ECO007 | English | done
ECO007 | Nepali | done
ECO008 | English | done
ECO008 | Nepali | done
ECO009 | English | done
ECO009 | Nepali | done
ECO010 | English | done
ECO010 | Nepali | done
ECO011 | English | done
ECO011 | Nepali | done
ECO012 | English | done
ECO012 | Nepali | done
ECO013 | English | done
ECO013 | Nepali | done
ECO014 | English | done
ECO014 | Nepali | done
ECO015 | English | done
ECO015 | Nepali | done
ECO016 | English | done
ECO016 | Nepali | done
ECO017 | English | done
ECO017 | Nepali | done
ECO018 | English | done
ECO018 | Nepali | done
ECO019 | English | done
ECO019 | Nepali | done
ECO020 | English | done
ECO020 | Nepali | done
ECO021 | English | done
ECO021 | Nepali | done
ECO022 | English | done
ECO022 | Nepali | done
ECO023 | English | d

In [11]:
results_df = pd.DataFrame(all_results)

output_file = os.path.join(
    folder,
    "Economy_Technology_Miscellaneous_Gemma3_1B_Results.csv"
)

results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(output_file)

print("\nRows:", len(results_df))
print("Expected:", len(df) * 2)

Saved:
C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research\Economy_Technology_Miscellaneous_Gemma3_1B_Results.csv

Rows: 266
Expected: 266
